# Рекурсия, lambda, pipeline

**Пара КТП 7** (2 ч) — введение рекурсии + отработка lambda/pipeline.

Минимум сдачи: flatten, walk_categories, A1, B1–B2, C1.

**Данные:** `NESTED_LIST` / `CATEGORY_TREE` ниже; далее — импорт из `module_datasets`.

In [ ]:
# Вложенный список (без dict)
NESTED_LIST = [1, [2, [3, 4]], 5]
# Простое дерево: (name, children)
CATEGORY_TREE = (
    'root',
    [
        ('electronics', [('phones', []), ('laptops', [])]),
        ('books', []),
    ],
)


## 1. flatten

Базовый случай: элемент **не** список. Шаг: если список — обойти элементы рекурсивно.

In [ ]:
def flatten(nested):
    """Список любой вложенности → плоский список."""
    pass


assert flatten(NESTED_LIST) == [1, 2, 3, 4, 5]

## 2. walk_categories

Дерево `(name, children)`. Печать с отступом `depth` пробелов.

In [ ]:
def walk_categories(node, depth=0):
    pass


# walk_categories(CATEGORY_TREE)


In [ ]:
import importlib.util
import sys
import urllib.request
from pathlib import Path

_RAW = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_01_functions_recursion/data/module_datasets.py"
)


def _import_module_datasets():
    for root in (Path("../..").resolve(), Path(".").resolve()):
        path = root / "data" / "module_datasets.py"
        if path.is_file():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            import data.module_datasets as md
            return md
    dest = Path("module_datasets.py")
    urllib.request.urlretrieve(_RAW, dest)
    spec = importlib.util.spec_from_file_location("module_datasets", dest)
    md = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(md)
    return md


_md = _import_module_datasets()
APARTMENTS = _md.APARTMENTS
EXAM_SCORES = _md.EXAM_SCORES
PREDICTIONS = _md.PREDICTIONS
LABELS = _md.LABELS
NESTED_API_RESPONSE = _md.NESTED_API_RESPONSE
CATEGORY_TREE = _md.CATEGORY_TREE
FEATURE_ROWS = _md.FEATURE_ROWS
MODEL_RUNS = _md.MODEL_RUNS
FEATURE_POINTS = _md.FEATURE_POINTS
PRICE_INTERCEPT = _md.PRICE_INTERCEPT
PRICE_COEF_AREA = _md.PRICE_COEF_AREA


---

# A. Рекурсия (серия)


## A1. `extract_ids`

In [ ]:
def extract_ids(nested):
    pass


assert extract_ids(NESTED_API_RESPONSE) == [1, 2, 3]


## A3 (углубление / ДЗ). `count_leaves`

In [ ]:
def count_leaves(node):
    pass


# assert count_leaves(CATEGORY_TREE) == 3


---

# B. Lambda


## B1. `filter` — аномалии (свой порог)

In [ ]:
anomalies = list(filter(lambda s: s < 50 or s > 90, EXAM_SCORES))
assert set(anomalies) == {40, 48, 91, 95}
print(anomalies)

## B2. Leaderboard `MODEL_RUNS`

Сортировка: f1 ↓; при равенстве — id ↑.

In [ ]:
leaderboard = sorted(MODEL_RUNS, key=lambda r: (-r['f1'], r['id']))
assert [r['id'] for r in leaderboard] == [25, 30, 305, 101, 200]
print([r['id'] for r in leaderboard])


## B3 (углубление / ДЗ). Farthest point

In [ ]:
# farthest = max(FEATURE_POINTS, key=lambda p: ...)
# assert farthest == [55, 12]
# print(farthest)


---

# C. Pipeline


## C1. `apply_pipeline`

In [ ]:
def apply_pipeline(data, steps):
    result = data
    for step in steps:
        result = step(result)
    return result


## C1. Цепочка для одной квартиры

`dict → area → scale → predict`

In [ ]:
def extract_area(row):
    return row['area_sqm']


def scale_area(area, max_area=60):
    return area / max_area


def predict_from_scaled(scaled_area):
    return PRICE_INTERCEPT + PRICE_COEF_AREA * (scaled_area * 60)


apt_pipeline = [extract_area, scale_area, predict_from_scaled]
pred = apply_pipeline(FEATURE_ROWS[0], apt_pipeline)
assert abs(pred - (PRICE_INTERCEPT + PRICE_COEF_AREA * 28)) < 1e-9
print('predicted mln:', pred)

## Эксперимент: порядок шагов

Что будет, если вызвать scale до extract? Запишите ошибку/вывод.

In [ ]:
# эксперимент


## C2 (углубление / ДЗ). Текстовый pipeline

In [ ]:
text_pipeline = [
    lambda s: s.strip().lower(),
    lambda s: s.split(),
    lambda tokens: [t for t in tokens if len(t) >= 4],
]

assert apply_pipeline('  Data science ML course  ', text_pipeline) == ['data', 'science', 'course']
print(apply_pipeline('  Data science ML course  ', text_pipeline))

## Мост к артефакту

Пара 8: [artifact/PROJECT.md](../../artifact/PROJECT.md) и [LESSON пары 8](../08_artifact/LESSON.md).

## Итог

**Pipeline = композиция функций.** Дальше — итоговый модуль `text_stats`.